# Project 05: Build a Small RAG Pipeline with Gemini

Build a short RAG pipeline. It loads a text file, splits it into chunks, stores the chunks in FAISS, retrieves the closest chunks for a question, and asks Gemini to answer from those chunks only.

Run top to bottom. Each step matches the README.

In [ ]:
import os, subprocess
REPO = "TharunPatel20/GenAI-Engineer"   # replace with your GitHub repo, e.g. "your-name/ai-engineer-course"
PROJECT = "projects/05-rag-fundamentals"
if REPO.startswith("<"):
    print("Repo not set. Upload the project files with the Files panel (left sidebar), into a folder named 05-rag-fundamentals, then run os.chdir('05-rag-fundamentals') before Step 1.")
else:
    subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git"], check=True)
    os.chdir(REPO.split("/")[1] + "/" + PROJECT)

## Step 1: install the pinned packages

This cell installs the exact versions from `requirements.txt`. Colab may ask you to restart the session. If it does, click **Restart session**, then run the cells again from the top (Step 0 onward). faiss-cpu is available on Colab, so it installs without a special build.

In [ ]:
%pip install -q langchain==0.3.27 langchain-community==0.3.27 langchain-core==0.3.72 langchain-google-genai==2.1.8 langchain-text-splitters==0.3.9 pypdf==5.6.1 faiss-cpu==1.11.0 python-dotenv==1.1.1

## Step 2: API key (Colab Secrets)

1. Click the key icon in the left sidebar (**Secrets**).
2. Click **Add new secret**. Name it exactly `GEMINI_API_KEY`, and paste your key as the value.
3. Turn on **Notebook access** for that secret.

Never paste the key into a cell. The cell below reads it from Colab Secrets.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception as e:
    print("No Colab secret found. Add GEMINI_API_KEY in Colab Secrets, then run this cell again. Details:", e)
print("key found:", bool(os.environ.get("GEMINI_API_KEY")))

### Step 1: Load the document
**What:** Read `sample_docs/company_notes.txt` into a LangChain `Document`.
**Why:** The pipeline needs text before it can do anything else. A document loader hides the file reading. PDF, web, and spreadsheet loaders work the same way in later projects.

This cell also sets the project constants and defines the `get_api_key()` helper that the later steps use.

In [ ]:
import os, sys

from langchain_community.document_loaders import TextLoader

DOC_PATH = "sample_docs/company_notes.txt"
INDEX_DIR = "faiss_index"

# Chunk settings. The stretch exercise asks you to try 300 and compare.
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# Model names change. Check Google AI Studio or the Gemini API docs if this fails.
EMBEDDING_MODEL = "models/gemini-embedding-001"
CHAT_MODEL = "gemini-2.5-flash"

TOP_K = 3  # how many chunks to retrieve

DEFAULT_QUESTION = "How much will Acme Robotics invest in AI in 2026, and where will the money go?"


def get_api_key():
    api_key = os.getenv("GEMINI_API_KEY", "").strip()
    if not api_key or api_key == "your-key-here":
        print("No Gemini API key found.")
        print("1. Create a free key in Google AI Studio (aistudio.google.com, 'Get API key').")
        print("2. Add GEMINI_API_KEY in Colab Secrets (key icon in the left sidebar), then run Step 2 again.")
        print("3. Run this cell and the next cells again.")
        sys.exit(1)
    return api_key


loader = TextLoader(DOC_PATH, encoding="utf-8")
documents = loader.load()
print(f"Loaded {len(documents)} document from {os.path.basename(DOC_PATH)}")

### Step 2: Split the text into chunks
**What:** Split the document into pieces of up to 1000 characters, with 100 characters of overlap between neighbors.
**Why:** An embedding represents one piece of text as one vector. A whole document makes a vague vector, and a model's context window is limited. Smaller chunks give more precise matches. The overlap keeps a sentence that crosses a boundary available in both chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks (chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

### Step 3: Embed the chunks and store them in FAISS
**What:** Send each chunk to Gemini's embedding model (`models/gemini-embedding-001`). Store the vectors in a FAISS index and save it to a folder named `faiss_index/`.
**Why:** An embedding turns meaning into numbers, so similar text gets similar vectors. FAISS is a fast local store for vectors. Saving the index means you do not pay to re-embed the document every time you ask a question.

Steps 1 to 3 together are the same as running `python build_index.py` on your computer. Run them before Step 4.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

api_key = get_api_key()
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL, google_api_key=api_key)
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local(INDEX_DIR)
print(f"Saved index with {len(chunks)} chunks to {INDEX_DIR}/")

### Step 4: Retrieve the closest chunks
**What:** Load the saved index, embed the question with the same embedding model, and ask FAISS for the 3 closest chunks with `similarity_search_with_score(question, k=3)`.
**Why:** Retrieval decides what the model is allowed to read. If the right chunk is not retrieved, the answer will be wrong no matter how good the model is. Always embed the question with the same model you used for the chunks. Comparing vectors from different models is like comparing kilograms with pounds.

**Score direction (read this).** The score is a number from the library, and its meaning depends on the library and the distance setting. FAISS with its default L2 (Euclidean) distance returns a distance, where a lower number means a closer match. Other setups, such as cosine similarity, can work the other way round. **Do not assume that a higher score is better.** Check the LangChain and FAISS docs for your installed version before you read meaning into a score. The cell prints a neutral label for this reason.

To ask your own question, change `question` below. Keep it in quotes.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

api_key = get_api_key()
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL, google_api_key=api_key)
# allow_dangerous_deserialization is needed because FAISS saves a pickle file.
# Only load an index that you built yourself in Step 3.
vector_store = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True,
)

question = DEFAULT_QUESTION
results = vector_store.similarity_search_with_score(question, k=TOP_K)

print(f"Question: {question}")
print()
print(f"Top {TOP_K} chunks with scores (check the score direction in the LangChain and FAISS docs):")
chunks = []
for number, (chunk, score) in enumerate(results, start=1):
    chunks.append(chunk)
    preview = chunk.page_content[:150].replace("\n", " ")
    print(f"  [{number}] score={score:.4f}  {preview}...")
print()

### Step 5: Generate a grounded answer
**What:** Put the retrieved chunks into a prompt with rules: answer only from the context, say "I don't know" if the answer is missing, and give supporting evidence. Send the prompt to `gemini-2.5-flash`.
**Why:** This is the "augmented" part of RAG. Grounding limits hallucination, and the evidence lets a person check the answer against the source.

Steps 4 and 5 together are the same as running `python ask_rag.py` on your computer.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Step 5: Build a grounded prompt. The model sees only the retrieved chunks.
def build_prompt(question, chunks):
    context = "\n\n".join(
        f"[Chunk {number}]\n{chunk.page_content}"
        for number, chunk in enumerate(chunks, start=1)
    )
    return (
        "You are a careful analyst. Answer the question using ONLY the context below.\n"
        "Rules:\n"
        "1. If the answer is not in the context, say \"I don't know\" and stop.\n"
        "2. After the answer, give supporting evidence: short quotes with their chunk numbers.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )


llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, google_api_key=api_key)
reply = llm.invoke(build_prompt(question, chunks))
print("Answer:")
print(reply.content)

## Stretch: chunk size experiment

This is the main exercise. It has no new code.

1. In the Step 1 cell, change `CHUNK_SIZE` to 300 and `CHUNK_OVERLAP` to 30.
2. Run Steps 2 and 3 again, then ask the same three questions with Steps 4 and 5 (change `question` each time).
3. Record the chunk count, the three chunk previews, and the answers.
4. Set the size back to 1000 and compare. Which setting gave the cleaner evidence? Which gave more context?

The other stretch goals (a two-part question, your own document, and a score threshold) are in the README.

## Checkpoints and what you learned

- After Step 1: `Loaded 1 document` (the text file loads as one document)
- After Step 2: a chunk count of 1 or more. Write it down. You will compare it in the stretch exercise
- After Step 3: a new `faiss_index/` folder appears in the Colab file browser (left sidebar)
- After Step 4: three chunks printed with scores. The first chunk should mention AI investment
- After Step 5: an answer that matches `company_notes.txt`, with quoted evidence
- Missing-answer test: set `question = "Who is the CEO of Acme Robotics?"`, then run Steps 4 and 5. The file does not say, so the answer should be "I don't know"

You learned how to load a document with a LangChain document loader, split text into overlapping chunks, create embeddings and store them in a FAISS vector store, retrieve the closest chunks and read scores with care, and write a grounded prompt that allows "I don't know". You also saw why lost-in-the-middle makes retrieval quality matter.

Run locally instead? See the README setup section.